# Step4 — Raw Zarr and signal processing

This notebook provides an interactive, one-recording test followed by two independent batch phases.

1. **Test one `stem`**: inspect raw BMP files, create a raw Zarr store, inspect masks, calculate signals, and review QC outputs.
2. **Batch Phase A**: create raw Zarr stores for all selected recordings.
3. **Batch Phase B**: process those raw Zarr stores into spatial signals.
4. **Optional Napari review**: inspect a manually selected recording outside batch loops.

The default trace ROI is derived from the brain and excluded-pixel masks, as selected by `signal_processing.roi.signal_roi`. This notebook does not provide a freehand ROI drawing step.

**Prerequisites:** Step1 `manifest_recordings.csv`, Step2 `metadata_registration.csv` and canonical masks, the raw BMP recordings, and the coefficient/pathlength tables required by the selected hemoglobin configuration.

**Important:** Step4 uses each recording's own spatial geometry; this notebook does not apply Step2/Step3 inter-recording registration to the raw images. Reference BMPs and masks from earlier stages must still be consistent with the configured raw rotation and scale.


## 0. Imports and local `src/` discovery

In [ ]:
import sys
from pathlib import Path

# Expected project layout:
# project/notebook/Step4_SignalProcessing.ipynb
# project/src/common.py
# project/src/step4_signal_processing.py

# Run the Jupyter kernel with project/notebook/ as its working directory.
NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = (NOTEBOOK_DIR.parent / "src").resolve()

required_files = ("common.py", "step4_signal_processing.py")
missing_files = [name for name in required_files if not (SRC_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Could not find the required source files: {', '.join(missing_files)}. "
        f"Expected location: {SRC_DIR}. "
        "Run this notebook from the project's notebook/ directory."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
import gc
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from step4_signal_processing import *

pd.set_option('display.max_columns', 120)
print('Source folder:', SRC_DIR)

## 1. Configuration path

Set `CONFIG_PATH` in the next cell to the YAML file used for this analysis. Its machine-specific path below is an **example from the original experiment**, not a required location. Keep a copy of the final working YAML together with the analysis for reproducibility; storing it under `project.analysis_parent_dir` is recommended but not enforced.

The configuration must contain the `step4` block (not an old `step5` block). The code does not use `step5` as a fallback.

Review `raw`, `raw_motion_correction`, `step4`, `signal_processing`, and `path_resolving`, especially wavelength order, sampling rate, mask locations, optical coefficient files, raw and processed overwrite settings, and the requested signal outputs.


In [ ]:
CONFIG_PATH = Path(r"config.yaml")
CONFIG_PATH

## 2. Build the Step4 index

`build_step4_index` reads the Step1 `manifest_recordings.csv` and Step2 `metadata_registration.csv`. It selects recording identities by `stem`, combines metadata with the authoritative raw-image directories from Step1, and resolves paths for raw and processed Zarr stores.

The table includes `group_id`, `ref_stem`, QC indicators, and existing-store flags when available. No raw signal processing takes place in this section.


In [ ]:
df_step4, config, paths = build_step4_index(CONFIG_PATH)
print('n recordings:', len(df_step4))
print('step4_root:', paths['step4_root'])
cols = [c for c in [
    'stem', 'group_id', 'ref_stem', 'is_bad',
    'img_dir_resolved', 'raw_zarr_exists', 'processed_zarr_exists',
    'raw_zarr_path_resolved', 'processed_zarr_path_resolved'
] if c in df_step4.columns]
df_step4[cols].head(20)

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df_step4[cols])

## 3. Optional selection for this notebook session

Use `SELECTED_GROUPS`, `SELECTED_STEMS`, and `INCLUDE_BAD` to narrow the index without modifying the YAML file. The example mouse and recording names in the next cell are **illustrative only**; replace them with IDs from your own manifest. `INCLUDE_BAD = None` delegates to `step4.review.include_bad`.


In [ ]:
SELECTED_GROUPS = None     # example: ['m206']
SELECTED_STEMS = None      # example: ['m206_Day1_Awake_Rest-state']
INCLUDE_BAD = None         # None = use step4.review.include_bad from config

df_step4, config, paths = build_step4_index(
    CONFIG_PATH,
    include_bad=INCLUDE_BAD,
    groups=SELECTED_GROUPS,
    stems=SELECTED_STEMS,
)
print('n selected:', len(df_step4))
df_step4[cols].head(20)

## 4. Select the test recording

Run a single recording end-to-end before starting the batch phases. The next cell uses `step4.notebook.preview_stem` when it is set.


In [ ]:
# Priority:
# 1) step4.notebook.preview_stem from config
# 2) sixteenth row (iloc[15]) of current df_step4
preview_stem = (config.get('step4', {}).get('notebook', {}) or {}).get('preview_stem')
if preview_stem:
    row_test = df_step4[df_step4['stem'].astype(str).eq(str(preview_stem))].iloc[0]
else:
    row_test = df_step4.iloc[0]

print('TEST STEM:', row_test['stem'])
print('GROUP:', row_test.get('group_id', ''))
print('IMG DIR:', row_test.get('img_dir_resolved', ''))
print('RAW ZARR:', row_test.get('raw_zarr_path_resolved', ''))
print('PROCESSED:', row_test.get('processed_zarr_path_resolved', ''))

# One-recording test — processing and QC

## 5. Inspect source frames and wavelength splitting

Confirm that the BMP naming pattern, the acquisition order in `raw.wave_pattern`, and `raw.reference_wave` match the recording. The resampling mode is configured by `raw.resample_mode`. The counts shown below allow a preliminary check before storing data.


In [ ]:
raw_info = inspect_raw_files_for_row(row_test, config)
print('n source files:', raw_info['n_source_files'])
print('reference wave:', raw_info['reference_wave'])
print('n time after resampling:', raw_info['n_time'])
for wave, paths_wave in raw_info['wave_to_paths_raw'].items():
    print(f'{wave}: raw={len(paths_wave)} resampled={len(raw_info["wave_to_paths_resampled"][wave])}')

## 6. Preview raw BMP channels before writing Zarr

The next cell displays representative images for the configured wavelength channels. Check channel ordering and image orientation before proceeding. `raw.rotation` controls rotation, while `raw.scale` applies to the raw Zarr geometry rather than the full-resolution reference images.


In [ ]:
fig = preview_raw_frames_from_bmp(row_test, config)
plt.show()

## 7. Build a raw Zarr store for the test recording

The raw-build function reads and resamples BMP frames in batches. It saves the `raw.zarr_key` array with wavelength and time coordinates and records per-pixel source intensity minima/maxima for later validity checks. `step4.overwrite_raw` determines whether an existing output may be replaced.


In [ ]:
raw_path = save_raw_zarr_for_row_batch(
    row_test,
    config,
    overwrite=config['step4'].get('overwrite_raw', False),
    show_progress=True,
)
raw_path

## 8. Reopen raw Zarr and review the stored channels

In [ ]:
raw = open_raw_zarr(row_test['raw_zarr_path_resolved'], config)
print(raw)
fig = preview_raw_frames_from_zarr(row_test, config)
plt.show()

## 9. Load masks and select ROIs

`load_masks_for_row` maps the saved canonical masks to the raw spatial shape and incorporates the all-frame pixel-validity mask derived from `signal_processing.raw_valid` thresholds.

The signal ROI is selected by `signal_processing.roi.signal_roi` (the example configuration uses `brain_signal`, based on brain and excluded-pixel masks). The drift-correction background ROI is selected by `signal_processing.roi.background_roi` when `signal_processing.drift_correction.enabled` is true. Check pixel counts and the mask preview before processing.


In [ ]:
raw_valid = load_raw_valid(raw_path, image_shape=(raw.sizes["y"], raw.sizes["x"]), config=config)
masks = load_masks_for_row(
    row_test,
    config,
    image_shape=(raw.sizes["y"], raw.sizes["x"]),
    raw_valid=raw_valid,
)
print("raw_valid fraction:", float(raw_valid.mean().compute().item()))

signal_roi = make_signal_roi(masks, config)
drift_enabled = bool(cfg_get(config, "signal_processing", "drift_correction", "enabled", default=True))
background_roi = make_background_roi(masks, config) if drift_enabled else None

print("brain source pixels:", int(masks["brain"].sum().compute().item()))
print("bad pixels:", int(masks["bad"].sum().compute().item()))
print("bregma pixels:", int(masks["bregma"].sum().compute().item()))
print("vessels pixels:", int(masks["vessels"].sum().compute().item()))
print("brainvessels_signal pixels:", int(masks["brainvessels_signal"].sum().compute().item()))
print("brain_signal pixels:", int(masks["brain_signal"].sum().compute().item()))
print("vessels_signal pixels:", int(masks["vessels_signal"].sum().compute().item()))
print("background pixels:", int(masks["background"].sum().compute().item()))
print("selected signal ROI pixels:", int(signal_roi.sum().compute().item()))
print("selected background ROI pixels:",
      int(background_roi.sum().compute().item()) if background_roi is not None else "not required (LED drift OFF)")

print("selected signal ROI mode:", cfg_get(config, "signal_processing", "roi", "signal_roi", default="brain_signal"))
print("selected background ROI mode:", cfg_get(config, "signal_processing", "roi", "background_roi", default="background"))

fig = preview_masks(row_test, masks, signal_roi=signal_roi)
plt.show()

## 10. Calculate and store processed signals for the test recording

`process_raw_zarr_for_row` applies the configured processing order, including optional drift correction, baseline normalization, reflectance-to-hemoglobin conversion and configured correction variants. The function writes a processed Zarr store and can generate processing and QC reports.

Static QC PNG saving depends on `signal_processing.output.save_static_qc_png` and other relevant flags; it is **not unconditional**. The next cell requests a QC preview for the selected recording. The saved biological signal names are selected by `signal_processing.output.save_signals`.


In [ ]:
processed_path = process_raw_zarr_for_row(
    row_test,
    config,
    overwrite=config["step4"].get("overwrite_processed", False),
    save_qc=True,
    show_progress=True,
    debug=True,
    save_step_report=True,
)
processed_path

In [ ]:
fig_frames, fig_traces = preview_processed_qc_for_row(row_test, config)
if fig_frames is not None:
    display(fig_frames)
display(fig_traces)


## 11. Inspect the processed Zarr and optical coefficients

Review the list of saved variables and their units. When hemoglobin inversion is enabled, the implementation stores the extinction coefficients and optical pathlength values actually used in the dataset attribute `hb_coefficients_used_json`.

The table displays wavelengths in nm, extinction-table values in `cm^-1 M^-1` (decadic convention), and pathlengths in mm. The inversion converts coefficients to natural-log units using `ln(10)` and pathlengths to cm. Concentration units for saved maps depend on `signal_processing.hemoglobin.scale_to_uM` and the actual function version used at runtime (see Step4 review notes for the CLI/import ordering issue).


In [ ]:
ds = xr.open_zarr(processed_path, chunks='auto', consolidated=False, mask_and_scale=False)
print(ds)
print('Variables:', list(ds.data_vars))
# Coefficients actually used in Hb inversion (saved in processed Zarr metadata).
import json
hb_used_json = ds.attrs.get("hb_coefficients_used_json")
if hb_used_json:
    hb_used = json.loads(hb_used_json)
    print("\nHb coefficients used in this processed Zarr:")
    print("Extinction table:", hb_used["extinction_table"])
    print("Pathlength table:", hb_used["pathlength_table"])
    display(pd.DataFrame(hb_used["wavelengths"])[
        ["wavelength_nm", "HbO_table", "HbR_table", "pathlength_mm",
         "extinction_method", "pathlength_method"]
    ])
    print("Table units: extinction cm^-1 M^-1; pathlength mm.")
    print("For inversion: extinction × ln(10), pathlength mm × 0.1 → cm.")
else:
    print("Hb coefficients not recorded in this processed Zarr (reprocess this stem to add metadata).")


## 12. Plot ROI time courses

The selected processed spatial signals are averaged over the chosen ROI and displayed versus the stored time coordinate. `step4.visualization.traces` controls the plotting layout and labels without changing the saved signal values.


In [ ]:
processed_masks = load_masks_for_row(
    row_test, config, image_shape=(ds.sizes["y"], ds.sizes["x"]),
    raw_valid=(ds["raw_valid"] == 1),
)
processed_signal_roi = make_signal_roi(processed_masks, config)
traces = dataset_roi_traces(ds, processed_signal_roi, max_vars=config['signal_processing']['output'].get('save_signals'))
display(traces.head())
fig = plot_roi_traces_dual_axis(traces, title=f"ROI traces: {row_test['stem']}", config=config)
plt.show()


## 13. Preview processed signal maps at a configured frame

In [ ]:
fig = preview_processed_frames(row_test, ds, config)
if fig is not None:
    plt.show()
else:
    print('No 3D processed variables selected for frame preview.')

# Batch execution

## 14. Batch Phase A — raw BMP to raw Zarr

Run this separately after validating the one-recording example. Phase A builds a raw store for each selected `stem` and logs its status. Existing complete stores can be reused when allowed by their completion markers and fingerprints. Once raw Zarr is available, later changes in baseline or signal regression do not necessarily require rereading BMP files.

The next cell defaults to `RUN_BATCH_RAW = True`: it **runs the batch** unless you explicitly disable it. Check the selected cohort and `step4.overwrite_raw` before execution.


In [ ]:
RUN_BATCH_RAW = True  # set True when test stem is OK

if RUN_BATCH_RAW:
    raw_log = run_raw_build_phase(
        df_step4,
        config,
        paths,
        overwrite=config['step4'].get('overwrite_raw', False),
        save_bmp_preview=True,
    )
    display(raw_log)
else:
    print('Batch raw build is disabled. Set RUN_BATCH_RAW=True to run.')

## 15. Refresh the index after Phase A

Rebuild the index to update the stored raw-Zarr existence flags before starting Phase B. Re-apply any intended selection of groups or stems; the selection variables in the next cell are set independently and default to `None`.


In [ ]:
SELECTED_GROUPS = None     # example: ['m206']
SELECTED_STEMS = None      # example: ['m206_Day1_Awake_Rest-state']
INCLUDE_BAD = None         # None = use step4.review.include_bad from config

df_step4, config, paths = build_step4_index(
    CONFIG_PATH,
    include_bad=INCLUDE_BAD,
    groups=SELECTED_GROUPS,
    stems=SELECTED_STEMS,
)
df_step4[cols].head(20)

## 16. Batch Phase B — raw Zarr to processed signals

Process each selected `stem` using the existing raw Zarr store. This phase does not keep all recordings loaded simultaneously. It writes processed datasets and logs errors without changing the underlying raw stores.

The next cell defaults to `RUN_BATCH_SIGNALS = True`: it **runs the batch** unless you disable it. Confirm the intended cohort and the `step4.overwrite_processed` setting first.


In [ ]:
RUN_BATCH_SIGNALS = True  # set True after raw zarr phase is OK

if RUN_BATCH_SIGNALS:
    signal_log = run_signal_processing_phase(
        df_step4,
        config,
        paths,
        overwrite=config['step4'].get('overwrite_processed', False),
    )
    display(signal_log)
else:
    print('Batch signal processing is disabled. Set RUN_BATCH_SIGNALS=True to run.')

# Optional manual Napari inspection

Launch interactive Napari review for one selected recording outside the batch loops when static quality-control figures are insufficient. No interactive Napari cell is executed automatically in this notebook section.


# Expected outputs and intermediates

Depending on the configured output flags, Step4 writes or may write:

- `raw_zarr/<stem>.zarr`: raw recording, coordinates, and source-intensity statistics;
- `processed_signals/<stem>_processed.zarr`: processed datasets (or a variant-specific suffix when configured);
- `qc_previews/<stem>/`: optional saved quality-control images and processing reports;
- `roi_traces/`: ROI time-course CSV files when enabled or requested;
- `processing_log.csv`: per-recording phase status.

Full background-divided intensity arrays, intermediate I/I0 and attenuation maps, alternate smoothing outputs, and diagnostic arrays are not all saved by default. Their creation depends on `signal_processing.output`, `signal_processing.smoothing`, and `step4.debug`/`step4.process_debug`.


## Reuse, reproducibility, and limitations

- `excluded_stems` is a configurable list of exact recording identifiers. **Example:** the supplied PT2021 YAML excludes one recording; other projects should set exclusions according to their own manifest and QC decisions.
- **Standard reuse:** Existing raw and processed Zarr stores must pass completion and fingerprint checks before reuse. Set `step4.overwrite_raw` or `step4.overwrite_processed` deliberately when recomputation is required; restore the desired defaults afterward.
- **Reuse of existing raw Zarr:** Set `step4.reuse_existing_raw_for_processing: true` to process previously saved raw Zarr data without rechecking the original BMP file paths, sizes, or modification times. This is useful when source images have been moved or copied after raw Zarr creation. The pipeline still verifies Zarr completeness, saved raw configuration compatibility, and wavelength coordinates. This option does not rebuild raw Zarr or modify the stored raw pixels. The default is `false`.
- **CLI reuse option:** `--reuse-existing-raw` enables the same behavior when combined with `--signals-only`. It cannot be used independently or with `--raw-only`.
- The two batch phases are controlled separately by notebook flags `RUN_BATCH_RAW` and `RUN_BATCH_SIGNALS`. To reprocess signals from existing raw Zarr, disable the raw-building phase and enable signal processing.
- Step4 processes images in each recording's native rotated/scaled geometry, without applying the Step2/Step3 registration transforms across recording days.
- Use Napari in a separate interactive session rather than inside batch loops.
- Raw Zarr stores contain the configured raw array and source-image extrema under `source_intensity_stats/raw_min` and `source_intensity_stats/raw_max`. The derived `raw_valid` mask is saved in processed outputs and can be recomputed from those extrema without reopening all BMP frames.
- Disabling `signal_processing.drift_correction.enabled` removes the background ROI requirement but retains the configured baseline normalization.
- **Hemoglobin units:** Hemoglobin concentration changes are calculated using the configured extinction coefficients and optical pathlengths. Output scaling, including conversion to µM when configured, is applied consistently during notebook imports and direct CLI execution.
- **Reproducibility:** Preserve the final YAML configuration, optical coefficient tables, and processing outputs. When using `reuse_existing_raw_for_processing`, results are derived from the saved raw Zarr rather than revalidated against the current source BMP files.